In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import os
# Initialize Claude
llm = ChatAnthropic(
     api_key=os.getenv("ANTHROPIC_API_KEY"),
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_ENDPOINT")
)
print("Setup complete!")

/home/ubuntu/Desktop/GenAi/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete!


In [2]:
# GenAI Customer Service Bot - Simple Q&A

def genai_support_bot(question: str) -> str:
    """Simple GenAI bot - no memory, no tools, just Q&A"""
    messages = [
        SystemMessage(content="You are a helpful customer service assistant for ShopEasy e-commerce."),
        HumanMessage(content=question)
    ]
    response = llm.invoke(messages)
    return response.content

# Test the GenAI bot
print("=" * 60)
print("GenAI Bot Response:")
print("=" * 60)
print(genai_support_bot("What is the status of my order #12345?"))

GenAI Bot Response:
I'd be happy to help you check your order status! However, I don't have access to your order information or ShopEasy's order database.

To check your order #12345, you can:

1. **Check your email** - Look for an order confirmation or shipping notification
2. **Log into your ShopEasy account** - Go to "My Orders" or "Order History" section
3. **Visit the order tracking page** - Use your order number and email address
4. **Contact ShopEasy customer service directly** - Call their phone number or use their live chat feature

If you'd like, I can help you troubleshoot issues or answer general questions about ShopEasy's policies. Is there anything else I can assist with?


In [3]:
# Testing memory limitation
print("Turn 1:")
print(genai_support_bot("My name is Sarah and my order number is #12345"))

print("\n" + "=" * 60)
print("Turn 2:")
print(genai_support_bot("What is my name and order number?"))

Turn 1:
Hello Sarah! 👋 

Thank you for providing your order number **#12345**. I'm here to help you with any questions or concerns about your order.

How can I assist you today? For example, I can help you with:
- **Order status** - Where is your package?
- **Tracking information** - Real-time delivery updates
- **Returns or exchanges** - Help with returning or exchanging items
- **Billing questions** - Issues with charges or refunds
- **General inquiries** - Any other questions about your order

What would you like to know?

Turn 2:
I don't have access to any of your personal information, order history, or account details. I'm a general customer service assistant without the ability to retrieve or view individual customer data.

To help you with your account or orders, I'd need you to provide:
- Your name
- Your order number
- Any other relevant details about your inquiry

Once you share that information, I'd be happy to assist you with questions about your order, returns, shipping, o

In [4]:
# Simulated database (in real-world, this would be your actual database)
ORDERS_DB = {
    "12345": {
        "customer": "Sarah Johnson",
        "status": "Shipped",
        "items": ["Wireless Headphones", "Phone Case"],
        "tracking": "TRK-789456123",
        "delivery_date": "June 26, 2026"
    },
    "67890": {
        "customer": "Mike Chen",
        "status": "Processing",
        "items": ["Laptop Stand", "USB-C Hub"],
        "tracking": None,
        "delivery_date": "Pending"
    }
}

INVENTORY_DB = {
    "Wireless Headphones": {"stock": 45, "price": 79.99},
    "Phone Case": {"stock": 120, "price": 19.99},
    "Laptop Stand": {"stock": 0, "price": 49.99},  # Out of stock!
    "USB-C Hub": {"stock": 30, "price": 39.99}
}

print("Databases loaded!")

Databases loaded!


In [5]:
from langchain_core.tools import tool

# Define tools that the agent can use

@tool
def lookup_order(order_id: str) -> str:
    """Look up order details by order ID. Returns order status, items, and tracking info."""
    order_id = order_id.replace("#", "").strip()
    if order_id in ORDERS_DB:
        order = ORDERS_DB[order_id]
        return f"""Order #{order_id}:
- Customer: {order['customer']}
- Status: {order['status']}
- Items: {', '.join(order['items'])}
- Tracking: {order['tracking'] or 'Not yet available'}
- Expected Delivery: {order['delivery_date']}"""
    return f"Order #{order_id} not found in our system."

@tool
def check_inventory(product_name: str) -> str:
    """Check if a product is in stock and get its price."""
    for name, info in INVENTORY_DB.items():
        if product_name.lower() in name.lower():
            status = "In Stock" if info['stock'] > 0 else "Out of Stock"
            return f"{name}: {status} ({info['stock']} units) - ${info['price']}"
    return f"Product '{product_name}' not found."

@tool
def initiate_return(order_id: str, reason: str) -> str:
    """Initiate a return request for an order."""
    order_id = order_id.replace("#", "").strip()
    if order_id in ORDERS_DB:
        return f"Return request initiated for Order #{order_id}. Reason: {reason}. Return label will be emailed within 24 hours."
    return f"Cannot initiate return - Order #{order_id} not found."

tools = [lookup_order, check_inventory, initiate_return]
print("Tools defined:", [t.name for t in tools])

Tools defined: ['lookup_order', 'check_inventory', 'initiate_return']


In [6]:
from langchain.agents import create_agent

# Create the Agentic AI bot with tools and memory

system_prompt = """You are a helpful customer service agent for ShopEasy e-commerce.
    
You have access to tools to:
- Look up order status and tracking
- Check product inventory
- Initiate returns

Always use the appropriate tool to get real information. Be friendly and helpful.
If you need to look something up, do it before responding."""

# Create the agent using LangChain 1.x API
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

# Memory store - now uses HumanMessage and AIMessage format
chat_history = []

def agentic_support_bot(user_input: str) -> str:
    """Agentic bot with tools and memory"""
    global chat_history
    
    # Build messages list: system + chat history + current input
    messages = chat_history + [HumanMessage(content=user_input)]
    
    # Invoke the agent with messages
    result = agent.invoke({"messages": messages})
    
    # Extract the response (it's the last message in the result)
    response_text = result["messages"][-1].content
    
    # Update memory
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response_text))
    
    return response_text

print("Agentic bot ready!")

Agentic bot ready!


In [7]:
# Test 1: Order lookup (Tool Use)
print("=" * 60)
print("Customer: What is the status of my order #12345?")
print("=" * 60)
response = agentic_support_bot("What is the status of my order #12345?")
print("\nAgent Response:")
print(response)

Customer: What is the status of my order #12345?

Agent Response:
Great news! Here's the status of your order #12345:

**Order Status:** Shipped ✓

**Items:**
- Wireless Headphones
- Phone Case

**Tracking Number:** TRK-789456123

**Expected Delivery Date:** June 26, 2026

Your order is on its way! You can use the tracking number to monitor its progress. Is there anything else I can help you with?


In [8]:
# Test 3: Multi-step reasoning - Complex request
print("=" * 60)
print("Customer: I want to return the headphones. They don't fit well.")
print("=" * 60)
response = agentic_support_bot("I want to return the headphones. They don't fit well.")
print("\nAgent Response:")
print(response)

Customer: I want to return the headphones. They don't fit well.

Agent Response:
Perfect! I've initiated a return request for the wireless headphones in order #12345.

**Return Details:**
- **Reason:** Headphones don't fit well
- **Return Label:** Will be emailed to you within 24 hours

Once you receive the return label, you can:
1. Pack the headphones securely
2. Print the return label and attach it to the package
3. Drop it off at the carrier location

You'll receive a refund once we receive and inspect the returned items. Is there anything else I can help you with regarding this order?


In [9]:
# TODO: Ask the agent if the Laptop Stand is in stock
# Hint: The agent should use the check_inventory tool

response = agentic_support_bot("Is the Laptop Stand available?")
print(response)

Unfortunately, the **Laptop Stand is currently out of stock**. 

**Product Details:**
- **Price:** $49.99
- **Availability:** Out of Stock (0 units)

However, we expect it to be back in stock soon. Would you like me to help you with anything else, or would you be interested in a similar product that might be available?


In [10]:
# TODO: Create a discount code tool
DISCOUNT_CODES = {
    "SAVE10": {"discount": 10, "min_order": 50},
    "WELCOME20": {"discount": 20, "min_order": 100},
}

@tool
def validate_discount_code(code: str) -> str:
    """Validate a discount code and return the discount details."""
    # YOUR CODE HERE
    # 1. Check if code exists in DISCOUNT_CODES
    # 2. Return discount details or "Invalid code" message
    pass

# After creating the tool, add it to the tools list and recreate the agent